# Weight Sharing Within a Layer

## Libraries

In [14]:
import time
import numpy as np
import torch
import torch.nn.functional as F
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader

In [15]:
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True

## Settings

In [16]:
# Device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Hyperparameters
random_seed = 1
learning_rate = 0.1
num_epochs = 10
batch_size = 128

# Architecture
num_classes = 10

## MNIST DATASET

In [17]:
train_dataset = datasets.MNIST(root='data', 
                               train=True, 
                               transform=transforms.ToTensor(),
                               download=True)

test_dataset = datasets.MNIST(root='data', 
                              train=False, 
                              transform=transforms.ToTensor())

train_loader = DataLoader(dataset=train_dataset, 
                          batch_size=batch_size, 
                          shuffle=True)

test_loader = DataLoader(dataset=test_dataset, 
                         batch_size=batch_size, 
                         shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 54.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.66MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 15.0MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.41MB/s]


In [18]:
for images, labels in train_loader:  
    print('Image batch dimensions:', images.shape)
    print('Image label dimensions:', labels.shape)
    break

Image batch dimensions: torch.Size([128, 1, 28, 28])
Image label dimensions: torch.Size([128])


## Model

In [19]:
class ConvNet(torch.nn.Module):

    def __init__(self, num_classes):
        super(ConvNet, self).__init__()
        self.conv_1 = torch.nn.Conv2d(in_channels=1,
                                      out_channels=4,
                                      kernel_size=(3, 3),
                                      stride=(1, 1),
                                      padding=1)
        self.pool_1 = torch.nn.MaxPool2d(kernel_size=(2, 2),
                                         stride=(2, 2),
                                         padding=0)
        self.conv_2 = torch.nn.Conv2d(in_channels=4,
                                      out_channels=8,
                                      kernel_size=(3, 3),
                                      stride=(1, 1),
                                      padding=1)       
        self.pool_2 = torch.nn.MaxPool2d(kernel_size=(2, 2),
                                         stride=(2, 2),
                                         padding=0) 
                                         
        self.linear_1 = torch.nn.Linear(7*7*8, 1, bias=False)
        
        self.linear_1_bias = torch.nn.Parameter(torch.tensor(torch.zeros(num_classes), dtype=self.linear_1.weight.dtype))
        
    def forward(self, x):
        out = self.conv_1(x)
        out = F.relu(out)
        out = self.pool_1(out)

        out = self.conv_2(out)
        out = F.relu(out)
        out = self.pool_2(out)

        logits = self.linear_1(out.view(-1, 7*7*8))
        
        logits = logits + self.linear_1_bias
        
        probas = F.softmax(logits, dim=1)
        return logits, probas

    
torch.manual_seed(random_seed)
model = ConvNet(num_classes=num_classes)

model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)  

/tmp/ipykernel_36/1128498297.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.linear_1_bias = torch.nn.Parameter(torch.tensor(torch.zeros(num_classes), dtype=self.linear_1.weight.dtype))


## Training

In [20]:
def compute_accuracy(model, data_loader):
    correct_pred, num_examples = 0, 0
    for features, targets in data_loader:
        features = features.to(device)
        targets = targets.to(device)
        logits, probas = model(features)
        _, predicted_labels = torch.max(probas, 1)
        num_examples += targets.size(0)
        correct_pred += (predicted_labels == targets).sum()
    return correct_pred.float()/num_examples * 100
    

start_time = time.time()
for epoch in range(num_epochs):
    model = model.train()
    for batch_idx, (features, targets) in enumerate(train_loader):
        
        features = features.to(device)
        targets = targets.to(device)

        ### FORWARD AND BACK PROP
        logits, probas = model(features)
        cost = F.cross_entropy(logits, targets)
        optimizer.zero_grad()
        
        cost.backward()
        
        ### UPDATE MODEL PARAMETERS
        optimizer.step()
        
        ### LOGGING
        if not batch_idx % 50:
            print ('Epoch: %03d/%03d | Batch %03d/%03d | Cost: %.4f' 
                   %(epoch+1, num_epochs, batch_idx, 
                     len(train_loader), cost))
    
    model = model.eval()
    with torch.set_grad_enabled(False): 
        print('Epoch: %03d/%03d training accuracy: %.2f%%' % (
              epoch+1, num_epochs, 
              compute_accuracy(model, train_loader)))
    
    print('Time elapsed: %.2f min' % ((time.time() - start_time)/60))
    
print('Total Training Time: %.2f min' % ((time.time() - start_time)/60))

Epoch: 001/010 | Batch 000/469 | Cost: 2.3026
Epoch: 001/010 | Batch 050/469 | Cost: 2.3030
Epoch: 001/010 | Batch 100/469 | Cost: 2.3026
Epoch: 001/010 | Batch 150/469 | Cost: 2.3029
Epoch: 001/010 | Batch 200/469 | Cost: 2.3009
Epoch: 001/010 | Batch 250/469 | Cost: 2.3061
Epoch: 001/010 | Batch 300/469 | Cost: 2.3037
Epoch: 001/010 | Batch 350/469 | Cost: 2.3002
Epoch: 001/010 | Batch 400/469 | Cost: 2.3012
Epoch: 001/010 | Batch 450/469 | Cost: 2.3115
Epoch: 001/010 training accuracy: 11.24%
Time elapsed: 0.18 min
Epoch: 002/010 | Batch 000/469 | Cost: 2.3036
Epoch: 002/010 | Batch 050/469 | Cost: 2.3015
Epoch: 002/010 | Batch 100/469 | Cost: 2.3041
Epoch: 002/010 | Batch 150/469 | Cost: 2.3046
Epoch: 002/010 | Batch 200/469 | Cost: 2.2958
Epoch: 002/010 | Batch 250/469 | Cost: 2.3103
Epoch: 002/010 | Batch 300/469 | Cost: 2.3033
Epoch: 002/010 | Batch 350/469 | Cost: 2.2980
Epoch: 002/010 | Batch 400/469 | Cost: 2.3097
Epoch: 002/010 | Batch 450/469 | Cost: 2.3055
Epoch: 002/010 t

Check that bias units updated correctly (should be all different):

In [21]:
model.linear_1_bias

Parameter containing:
tensor([-0.0163,  0.1487,  0.0025,  0.0134,  0.0002, -0.1049, -0.0478,  0.0540,
        -0.0315, -0.0183], device='cuda:0', requires_grad=True)

## Evaluation

In [22]:
with torch.set_grad_enabled(False):
    print('Test accuracy: %.2f%%' % (compute_accuracy(model, test_loader)))

Test accuracy: 11.35%
